In [1]:
import torch
import torch.nn.functional as F
import math

# ============================================================
# NAIVE ATTENTION IMPLEMENTATION
# This is the textbook implementation
# It materializes the full [seq x seq] score matrix
# which is exactly what FlashAttention avoids
# ============================================================

torch.manual_seed(42)

batch_size = 1
num_heads = 1      # keeping it simple — 1 head to see clearly
seq_len = 512      # sequence length
head_dim = 64      # dimension per head

# Create Q, K, V tensors
# In a real transformer these come from linear projections of input
Q = torch.randn(batch_size, num_heads, seq_len, head_dim)
K = torch.randn(batch_size, num_heads, seq_len, head_dim)
V = torch.randn(batch_size, num_heads, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print()

# ============================================================
# STEP 1: Compute attention scores S = QKᵀ / sqrt(d)
# This creates the [seq x seq] matrix that is the problem
# ============================================================

scale = math.sqrt(head_dim)   # = sqrt(64) = 8.0

# Q @ K.transpose(-2,-1) means: for each head, multiply Q by Kᵀ
# Result shape: [batch x heads x seq x seq]
S = torch.matmul(Q, K.transpose(-2, -1)) / scale

print(f"Attention score matrix S shape: {S.shape}")
print(f"This matrix has {S.shape[-1] * S.shape[-2]:,} numbers")
print(f"Memory: {S.numel() * 2 / 1024:.1f} KB in float16")
print(f"For seq=4096 this would be: {4096*4096*2/1024/1024:.1f} MB — per head!")
print()

# ============================================================
# STEP 2: Softmax over last dimension
# Each row becomes a probability distribution (sums to 1)
# This requires seeing ALL scores in each row first
# ============================================================

P = F.softmax(S, dim=-1)   # shape: [batch x heads x seq x seq]

print(f"After softmax P shape: {P.shape}")
print(f"Each row sums to 1: {P[0,0,0].sum().item():.6f}")  # should be ~1.0
print()

# ============================================================
# STEP 3: Multiply by V to get weighted output
# Each output token = weighted sum of all V vectors
# weights come from softmax scores
# ============================================================

output_naive = torch.matmul(P, V)   # shape: [batch x heads x seq x head_dim]

print(f"Output shape: {output_naive.shape}")
print(f"\nSummary of what happened in memory:")
print(f"1. Computed S = QKᵀ — shape {S.shape} — stored in memory")
print(f"2. Computed P = softmax(S) — shape {P.shape} — stored in memory")
print(f"3. Computed output = PV — shape {output_naive.shape}")
print(f"The [seq x seq] matrix was written and read multiple times")
print(f"That's the bottleneck FlashAttention eliminates")

Q shape: torch.Size([1, 1, 512, 64])
K shape: torch.Size([1, 1, 512, 64])
V shape: torch.Size([1, 1, 512, 64])

Attention score matrix S shape: torch.Size([1, 1, 512, 512])
This matrix has 262,144 numbers
Memory: 512.0 KB in float16
For seq=4096 this would be: 32.0 MB — per head!

After softmax P shape: torch.Size([1, 1, 512, 512])
Each row sums to 1: 1.000000

Output shape: torch.Size([1, 1, 512, 64])

Summary of what happened in memory:
1. Computed S = QKᵀ — shape torch.Size([1, 1, 512, 512]) — stored in memory
2. Computed P = softmax(S) — shape torch.Size([1, 1, 512, 512]) — stored in memory
3. Computed output = PV — shape torch.Size([1, 1, 512, 64])
The [seq x seq] matrix was written and read multiple times
That's the bottleneck FlashAttention eliminates


In [2]:
import torch

# ============================================================
# UNDERSTANDING NUMERICAL STABILITY IN SOFTMAX
# This is the foundation of the running-max trick in FlashAttention
# ============================================================

# First — show why naive exp() is dangerous
print("=== WHY WE NEED THE MAX TRICK ===\n")

scores_dangerous = torch.tensor([100.0, 200.0, 300.0])
print(f"Scores: {scores_dangerous.tolist()}")

# Naive exp — will overflow
exp_naive = torch.exp(scores_dangerous)
print(f"exp(scores) naive: {exp_naive.tolist()}")
print(f"Problem: inf values! Softmax becomes nan\n")

# Stable exp — subtract max first
max_val = scores_dangerous.max()
exp_stable = torch.exp(scores_dangerous - max_val)
print(f"Subtract max ({max_val.item()}) first")
print(f"exp(scores - max): {exp_stable.tolist()}")
print(f"softmax = exp_stable / exp_stable.sum() = {(exp_stable / exp_stable.sum()).tolist()}")

# Verify same result as torch.softmax
softmax_result = torch.softmax(scores_dangerous, dim=0)
print(f"torch.softmax result: {softmax_result.tolist()}")
print(f"They match: {torch.allclose(exp_stable/exp_stable.sum(), softmax_result)}")

print("\n=== THE RUNNING MAX TRICK (how FlashAttention tiles softmax) ===\n")

# Simulate one row of attention scores split into 3 tiles
# as if we're processing them one at a time without seeing all at once
torch.manual_seed(42)
full_scores = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0, 5.0])

tile1 = full_scores[0:3]   # [3, 1, 4]
tile2 = full_scores[3:6]   # [1, 5, 9]
tile3 = full_scores[6:9]   # [2, 6, 5]

print(f"Full scores: {full_scores.tolist()}")
print(f"Tile 1: {tile1.tolist()}")
print(f"Tile 2: {tile2.tolist()}")
print(f"Tile 3: {tile3.tolist()}")
print()

# ============================================================
# Ground truth: compute softmax on full scores at once
# ============================================================
ground_truth_softmax = torch.softmax(full_scores, dim=0)
print(f"Ground truth softmax: {[round(v,4) for v in ground_truth_softmax.tolist()]}")
print()

# ============================================================
# Tiled computation with running max correction
# ============================================================

# Initialize running statistics
running_max = float('-inf')   # start with negative infinity
running_sum = 0.0             # denominator accumulator
running_numerator = torch.zeros(9)  # numerator accumulator (exp × value)

print("--- Processing Tile 1 ---")
tile1_max = tile1.max().item()
new_max = max(running_max, tile1_max)
print(f"Tile max: {tile1_max}, Running max updated to: {new_max}")

# No previous data to correct — just process tile 1
exp_tile1 = torch.exp(tile1 - new_max)
running_sum = exp_tile1.sum().item()
running_max = new_max

# Store the exp values in their positions
running_numerator[0:3] = exp_tile1
print(f"exp(tile1 - max): {[round(v,4) for v in exp_tile1.tolist()]}")
print(f"Running sum: {running_sum:.4f}")
print()

print("--- Processing Tile 2 ---")
tile2_max = tile2.max().item()
new_max = max(running_max, tile2_max)
print(f"Tile max: {tile2_max}, Running max updated to: {new_max}")

if new_max != running_max:
    # Max changed! Must correct previous accumulated values
    correction = math.exp(running_max - new_max)
    print(f"Max changed from {running_max} to {new_max}")
    print(f"Correction factor: exp({running_max} - {new_max}) = {correction:.6f}")

    # Scale down previous numerator and sum
    running_numerator[0:3] = running_numerator[0:3] * correction
    running_sum = running_sum * correction
    print(f"Corrected running sum: {running_sum:.6f}")

running_max = new_max
exp_tile2 = torch.exp(tile2 - running_max)
running_sum += exp_tile2.sum().item()
running_numerator[3:6] = exp_tile2
print(f"exp(tile2 - max): {[round(v,4) for v in exp_tile2.tolist()]}")
print(f"Running sum: {running_sum:.4f}")
print()

print("--- Processing Tile 3 ---")
tile3_max = tile3.max().item()
new_max = max(running_max, tile3_max)
print(f"Tile max: {tile3_max}, Running max stays: {new_max} (no change)")

exp_tile3 = torch.exp(tile3 - running_max)
running_sum += exp_tile3.sum().item()
running_numerator[6:9] = exp_tile3
print(f"exp(tile3 - max): {[round(v,4) for v in exp_tile3.tolist()]}")
print(f"Final running sum: {running_sum:.4f}")
print()

# ============================================================
# Final: divide numerator by running sum to get softmax
# ============================================================
tiled_softmax = running_numerator / running_sum

print("=== FINAL COMPARISON ===")
print(f"Ground truth softmax: {[round(v,4) for v in ground_truth_softmax.tolist()]}")
print(f"Tiled softmax result: {[round(v,4) for v in tiled_softmax.tolist()]}")
match = torch.allclose(ground_truth_softmax, tiled_softmax, atol=1e-5)
print(f"Results match: {match}")
print("✅ Tiled softmax with running max gives EXACT same result!" if match else "❌ Mismatch")
print("\nThis is the mathematical core of FlashAttention —")
print("correct softmax without ever seeing all scores at once!")

=== WHY WE NEED THE MAX TRICK ===

Scores: [100.0, 200.0, 300.0]
exp(scores) naive: [inf, inf, inf]
Problem: inf values! Softmax becomes nan

Subtract max (300.0) first
exp(scores - max): [0.0, 3.783505853677006e-44, 1.0]
softmax = exp_stable / exp_stable.sum() = [0.0, 3.783505853677006e-44, 1.0]
torch.softmax result: [0.0, 3.783505853677006e-44, 1.0]
They match: True

=== THE RUNNING MAX TRICK (how FlashAttention tiles softmax) ===

Full scores: [3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0, 5.0]
Tile 1: [3.0, 1.0, 4.0]
Tile 2: [1.0, 5.0, 9.0]
Tile 3: [2.0, 6.0, 5.0]

Ground truth softmax: [0.0023, 0.0003, 0.0061, 0.0003, 0.0167, 0.9114, 0.0008, 0.0454, 0.0167]

--- Processing Tile 1 ---
Tile max: 4.0, Running max updated to: 4.0
exp(tile1 - max): [0.3679, 0.0498, 1.0]
Running sum: 1.4177

--- Processing Tile 2 ---
Tile max: 9.0, Running max updated to: 9.0
Max changed from 4.0 to 9.0
Correction factor: exp(4.0 - 9.0) = 0.006738
Corrected running sum: 0.009552
exp(tile2 - max): [0.0003, 0.0

In [3]:
import torch
import torch.nn.functional as F
import math

# ============================================================
# FLASHATTENTION STYLE TILED ATTENTION
# Full simulation of the tiling approach
# We process Q and K/V in tiles
# Never materialize the full [seq x seq] matrix
# ============================================================

torch.manual_seed(42)

seq_len = 128    # sequence length
head_dim = 64    # dimension per attention head
tile_size = 32   # how big each tile is — in practice ~64-128

Q = torch.randn(seq_len, head_dim)
K = torch.randn(seq_len, head_dim)
V = torch.randn(seq_len, head_dim)

scale = math.sqrt(head_dim)

# Output accumulator — this is what we build up tile by tile
# Shape: [seq_len x head_dim]
output = torch.zeros(seq_len, head_dim)

# For each query position we maintain:
# running_max: the max score seen so far for this query
# running_sum: the sum of exp(score - max) seen so far
running_max = torch.full((seq_len,), float('-inf'))   # shape: [seq_len]
running_sum = torch.zeros(seq_len)                     # shape: [seq_len]

num_tiles = seq_len // tile_size
print(f"Sequence length: {seq_len}")
print(f"Tile size: {tile_size}")
print(f"Number of K/V tiles to process: {num_tiles}")
print(f"Max [seq x seq] matrix size we ever store: [{tile_size} x {tile_size}] = {tile_size*tile_size} numbers")
print(f"vs naive attention which stores: [{seq_len} x {seq_len}] = {seq_len*seq_len} numbers")
print(f"Memory reduction: {seq_len*seq_len / (tile_size*tile_size):.0f}x\n")

# ============================================================
# OUTER LOOP: iterate over K and V tiles
# Each tile is a chunk of key-value pairs
# ============================================================

for tile_idx in range(num_tiles):
    # Get current K and V tile
    start = tile_idx * tile_size
    end = start + tile_size

    K_tile = K[start:end, :]   # shape: [tile_size x head_dim]
    V_tile = V[start:end, :]   # shape: [tile_size x head_dim]

    # ============================================================
    # Compute scores between ALL queries and this tile of keys
    # Shape: [seq_len x tile_size]
    # This is small! Not [seq x seq] — just [seq x tile_size]
    # ============================================================

    scores_tile = (Q @ K_tile.T) / scale   # shape: [seq_len x tile_size]

    # ============================================================
    # Find max score in this tile for each query (for stability)
    # ============================================================

    tile_max = scores_tile.max(dim=1).values   # shape: [seq_len]

    # ============================================================
    # Update running max and compute correction factor
    # If new tile has higher scores, old accumulated output is wrong
    # We must correct it by scaling down by correction factor
    # ============================================================

    new_max = torch.maximum(running_max, tile_max)   # element-wise max

    # Correction factor for previously accumulated output
    # exp(old_max - new_max) rescales old values to new max reference
    correction = torch.exp(running_max - new_max)   # shape: [seq_len]

    # ============================================================
    # Scale down previous output and running sum by correction
    # This "undoes" the old max assumption and applies new max
    # ============================================================

    output = output * correction.unsqueeze(1)    # scale each row of output
    running_sum = running_sum * correction       # scale running sum

    # ============================================================
    # Compute exp of scores for this tile (using new max for stability)
    # ============================================================

    exp_scores = torch.exp(
        scores_tile - new_max.unsqueeze(1)   # subtract new_max from each row
    )   # shape: [seq_len x tile_size]

    # ============================================================
    # Accumulate: add this tile's contribution to output and sum
    # output += exp_scores @ V_tile  (weighted sum of V values)
    # running_sum += sum of exp_scores (for normalization later)
    # ============================================================

    output = output + exp_scores @ V_tile        # shape: [seq_len x head_dim]
    running_sum = running_sum + exp_scores.sum(dim=1)   # shape: [seq_len]

    # Update running max for next iteration
    running_max = new_max

# ============================================================
# Final normalization: divide accumulated output by running sum
# This is equivalent to dividing by the softmax denominator
# ============================================================

output = output / running_sum.unsqueeze(1)   # shape: [seq_len x head_dim]

print(f"FlashAttention-style output shape: {output.shape}")

# ============================================================
# VERIFY against naive attention
# Both must give identical results
# ============================================================

# Compute naive attention for comparison
scores_full = (Q @ K.T) / scale                    # [seq x seq]
attn_weights = F.softmax(scores_full, dim=-1)      # [seq x seq]
output_naive = attn_weights @ V                    # [seq x head_dim]

match = torch.allclose(output, output_naive, atol=1e-4)
print(f"Naive attention output shape:        {output_naive.shape}")
print(f"\nResults match: {match}")
print(" Tiled FlashAttention gives exact same result as naive!" if match else "❌ Mismatch")
print("\nBut FlashAttention never stored the full [128 x 128] score matrix")
print(f"It only ever stored tiles of size [{tile_size} x {tile_size}]")
print("That's the entire point!")

Sequence length: 128
Tile size: 32
Number of K/V tiles to process: 4
Max [seq x seq] matrix size we ever store: [32 x 32] = 1024 numbers
vs naive attention which stores: [128 x 128] = 16384 numbers
Memory reduction: 16x

FlashAttention-style output shape: torch.Size([128, 64])
Naive attention output shape:        torch.Size([128, 64])

Results match: True
 Tiled FlashAttention gives exact same result as naive!

But FlashAttention never stored the full [128 x 128] score matrix
It only ever stored tiles of size [32 x 32]
That's the entire point!


In [4]:
import torch
import torch.nn.functional as F
import math

# ============================================================
# EXPERIMENT: compare memory usage naive vs tiled attention
# Track how much memory the [seq x seq] matrix takes
# as sequence length grows — this shows WHY FlashAttention matters
# ============================================================

print("Memory usage: Naive Attention vs FlashAttention-style")
print("="*65)
print(f"{'Seq Len':<12} {'Naive [seq²] MB':<20} {'Tiled [max tile] MB':<22} {'Saving'}")
print("-"*65)

head_dim = 64
tile_size = 64   # FlashAttention uses fixed tile size regardless of seq length

seq_lengths = [256, 512, 1024, 2048, 4096, 8192, 16384]

for seq_len in seq_lengths:
    # Naive attention must store full [seq x seq] matrix
    # In float16, each number = 2 bytes
    naive_matrix_elements = seq_len * seq_len
    naive_mb = naive_matrix_elements * 2 / 1024 / 1024   # convert to MB

    # FlashAttention only needs tiles — max [seq_len x tile_size] at once
    # (for the intermediate score tile, not the full matrix)
    tiled_matrix_elements = seq_len * tile_size
    tiled_mb = tiled_matrix_elements * 2 / 1024 / 1024

    saving = naive_mb / tiled_mb

    print(f"{seq_len:<12} {naive_mb:<20.2f} {tiled_mb:<22.4f} {saving:.0f}x")

print()
print(" What this table shows:")
print("Naive attention memory grows with seq² (quadratically)")
print("FlashAttention memory grows with seq×tile_size (linearly)")
print("At seq=4096: naive needs 32MB, tiled needs only 0.5MB — 64x less")
print("At seq=16384: naive needs 512MB (!), tiled still only 2MB")
print()
print("This is why FlashAttention enables long context (32K, 128K tokens)")
print("Without it, you'd run out of GPU memory just for attention matrices")

Memory usage: Naive Attention vs FlashAttention-style
Seq Len      Naive [seq²] MB      Tiled [max tile] MB    Saving
-----------------------------------------------------------------
256          0.12                 0.0312                 4x
512          0.50                 0.0625                 8x
1024         2.00                 0.1250                 16x
2048         8.00                 0.2500                 32x
4096         32.00                0.5000                 64x
8192         128.00               1.0000                 128x
16384        512.00               2.0000                 256x

 What this table shows:
Naive attention memory grows with seq² (quadratically)
FlashAttention memory grows with seq×tile_size (linearly)
At seq=4096: naive needs 32MB, tiled needs only 0.5MB — 64x less
At seq=16384: naive needs 512MB (!), tiled still only 2MB

This is why FlashAttention enables long context (32K, 128K tokens)
Without it, you'd run out of GPU memory just for attention m